In [1]:
from typing import Optional, get_args, get_type_hints

from vectormesh.components import (
    Concatenate2D,
    MeanAggregator,
    NeuralNet,
    Parallel,
    Serial,
)


In [2]:
model = Serial([MeanAggregator(), NeuralNet(hidden_size=768, out_size=32)])

In [3]:
agg = MeanAggregator()

In [4]:
hint = get_type_hints(agg.forward)
hint

{'embeddings': jaxtyping.Float[Tensor, 'batch _ dim'],
 'return': jaxtyping.Float[Tensor, 'batch dim']}

In [11]:
def extract_shape_from_hint(hint) -> Optional[str]:
    """Extract shape string from jaxtyping Float[Tensor, "batch dim"] annotation.

    Args:
        hint: Type hint that may contain jaxtyping annotation

    Returns:
        Shape string like "batch dim" or None if not extractable
    """
    try:
        # Check if this is a jaxtyping annotation
        if hasattr(hint, "__origin__"):
            # For Float[Tensor, "batch dim"], get_args returns the Tensor and shape
            args = get_args(hint)
            if args and hasattr(args[0], "dim_str"):
                return args[0].dim_str
            # Try getting dim_str directly
            if hasattr(hint, "dim_str"):
                return hint.dim_str
    except Exception:
        pass
    return None

In [13]:
hints = get_type_hints(agg.forward)
extract_shape_from_hint(hints)

In [ ]:
parallel = Parallel(
    [
        # (batch, chunks, dims) -> (batch, dims) -> (batch, 32)
        Serial([MeanAggregator(), NeuralNet(hidden_size=768, out_size=32)]),
        # (batch, dims) -> (batch, 32)
        Serial([NeuralNet(hidden_size=123, out_size=32)]),
    ]
)

pipeline = Serial(
    [
        parallel,  # (X1, X2) -> (batch, 32), (batch, 32)
        Concatenate2D(),  # (batch, 32), (batch, 32) -> (batch, 64)
        NeuralNet(hidden_size=64, out_size=32),  # (batch, 64) -> (batch, 32)
    ]
)